In [1]:
#! pip install chromadb
#! pip install -U langchain langchain-community langchain-openai pydantic


In [2]:
from config_loader import *
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

In [3]:
from pathlib import Path
Path('./download/10k_html/MSFT-20250730.pdf').stem

'MSFT-20250730'

In [4]:
loader=PyMuPDFLoader(
    file_path='download\\10k_pdf\\MSFT-20250730.pdf'
)
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
docs=loader.load()
split_docs=splitter.split_documents(docs)
persist_dir = "chroma_store_large"
embed=OpenAIEmbeddings(model=os.environ['OPENAI_EMBEDED_MODEL'])
vector_store=Chroma.from_documents(
    documents=split_docs,
    embedding=embed,
    persist_directory=persist_dir,
    collection_name="MSFT"
    )


ValueError: File path download\10k_pdf\MSFT-20250730.pdf is not a valid file or url

Retriever

In [ ]:
retriever= vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":2}
    )
query="identify sentiments around AI as a growth driver vs a controlled risk"
result=retriever.invoke(query)
for i, doc in enumerate(result):
    print(f'\n----Result {i+1}-----\n')
    print(doc.page_content)



----Result 1-----

PART I
Item 1A
Issues in the development, deployment, and use of AI may result in reputational or competitive 
harm
 
or
 
liability. We are building AI into many of our offerings, including our productivity services, and we are also 
making AI available for our customers to use in solutions that they build. This AI may be developed by 
Microsoft or others, including our strategic partner, OpenAI. We expect these elements of our business to 
grow. We envision a future in which AI operating in devices, applications, and the cloud helps our  
customers be more productive in their work and personal lives. As with many innovations, AI presents 
risks and challenges that could affect its adoption, and therefore our business. AI algorithms or training 
methodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate 
information. Content generated by AI systems may be offensive, illegal, inaccurate, or otherwise harmful.

----Result 2-

In [ ]:
retriever2= vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":2}
    )
query="identify sentiments around AI as a growth driver vs a controlled risk"
result=retriever2.invoke(query)
for i, doc in enumerate(result):
    print(f'\n----Result {i+1}-----\n')
    print(doc.page_content)



----Result 1-----

PART I
Item 1A
Issues in the development, deployment, and use of AI may result in reputational or competitive 
harm
 
or
 
liability. We are building AI into many of our offerings, including our productivity services, and we are also 
making AI available for our customers to use in solutions that they build. This AI may be developed by 
Microsoft or others, including our strategic partner, OpenAI. We expect these elements of our business to 
grow. We envision a future in which AI operating in devices, applications, and the cloud helps our  
customers be more productive in their work and personal lives. As with many innovations, AI presents 
risks and challenges that could affect its adoption, and therefore our business. AI algorithms or training 
methodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate 
information. Content generated by AI systems may be offensive, illegal, inaccurate, or otherwise harmful.

----Result 2-

In [ ]:
retriever2= vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={'score_threshold': 0.1}
    
    )
query="identify sentiments around AI as a growth driver vs a controlled risk"
result=retriever2.invoke(query)
for i, doc in enumerate(result):
    print(f'\n----Result {i+1}-----\n')
    print(doc.page_content)



----Result 1-----

PART I
Item 1A
Issues in the development, deployment, and use of AI may result in reputational or competitive 
harm
 
or
 
liability. We are building AI into many of our offerings, including our productivity services, and we are also 
making AI available for our customers to use in solutions that they build. This AI may be developed by 
Microsoft or others, including our strategic partner, OpenAI. We expect these elements of our business to 
grow. We envision a future in which AI operating in devices, applications, and the cloud helps our  
customers be more productive in their work and personal lives. As with many innovations, AI presents 
risks and challenges that could affect its adoption, and therefore our business. AI algorithms or training 
methodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate 
information. Content generated by AI systems may be offensive, illegal, inaccurate, or otherwise harmful.

----Result 2-

In [ ]:
result

[Document(metadata={'author': '', 'creationDate': '', 'creationdate': '', 'creator': '', 'file_path': 'download\\10k_pdf\\MSFT-20250730.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': '', 'moddate': '', 'page': 39, 'producer': 'WeasyPrint 66.0', 'source': 'download\\10k_pdf\\MSFT-20250730.pdf', 'subject': '', 'title': '10-K', 'total_pages': 167, 'trapped': ''}, page_content='PART I\nItem 1A\nIssues in the development, deployment, and use of AI may result in reputational or competitive \nharm\n \nor\n \nliability. We are building AI into many of our offerings, including our productivity services, and we are also \nmaking AI available for our customers to use in solutions that they build. This AI may be developed by \nMicrosoft or others, including our strategic partner, OpenAI. We expect these elements of our business to \ngrow. We envision a future in which AI operating in devices, applications, and the cloud helps our  \ncustomers be more productive in their work and personal li

In [ ]:
query="identify sentiments around AI as a growth driver vs a controlled risk"
result=vector_store.similarity_search(query,k=5)
result[0].page_content

'PART I\nItem 1A\nIssues in the development, deployment, and use of AI may result in reputational or competitive \nharm\n \nor\n \nliability. We are building AI into many of our offerings, including our productivity services, and we are also \nmaking AI available for our customers to use in solutions that they build. This AI may be developed by \nMicrosoft or others, including our strategic partner, OpenAI. We expect these elements of our business to \ngrow. We envision a future in which AI operating in devices, applications, and the cloud helps our  \ncustomers be more productive in their work and personal lives. As with many innovations, AI presents \nrisks and challenges that could affect its adoption, and therefore our business. AI algorithms or training \nmethodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate \ninformation. Content generated by AI systems may be offensive, illegal, inaccurate, or otherwise harmful.'

In [ ]:
query="identify sentiments around AI as a growth driver vs a controlled risk"
result=vector_store.similarity_search_with_score(query,k=5)
result

[(Document(metadata={'author': '', 'creationDate': '', 'creationdate': '', 'creator': '', 'file_path': 'download\\10k_pdf\\MSFT-20250730.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': '', 'moddate': '', 'page': 39, 'producer': 'WeasyPrint 66.0', 'source': 'download\\10k_pdf\\MSFT-20250730.pdf', 'subject': '', 'title': '10-K', 'total_pages': 167, 'trapped': ''}, page_content='PART I\nItem 1A\nIssues in the development, deployment, and use of AI may result in reputational or competitive \nharm\n \nor\n \nliability. We are building AI into many of our offerings, including our productivity services, and we are also \nmaking AI available for our customers to use in solutions that they build. This AI may be developed by \nMicrosoft or others, including our strategic partner, OpenAI. We expect these elements of our business to \ngrow. We envision a future in which AI operating in devices, applications, and the cloud helps our  \ncustomers be more productive in their work and personal l

Multi query retrievers

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

model=ChatOpenAI(
    model=os.environ["OPENAI_MODEL"],
    temperature=0,
    max_completion_tokens=100)

base_retriever=vector_store.as_retriever(search_type='similarity',search_kwargs={'k':2})


prompt=PromptTemplate(
    template="""Please provide the information as a top leader and remove all noises from the answers .
      Query is {query}""",
      input_variables=[query])

query="identify sentiments around AI as a growth driver vs a controlled risk"

retriver=MultiQueryRetriever.from_llm(
    llm=model,
    retriever=base_retriever,
    prompt=prompt
)
result=retriever.invoke(query)
print(result)


[Document(metadata={'author': '', 'creationDate': '', 'creationdate': '', 'creator': '', 'file_path': 'download\\10k_pdf\\MSFT-20250730.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': '', 'moddate': '', 'page': 39, 'producer': 'WeasyPrint 66.0', 'source': 'download\\10k_pdf\\MSFT-20250730.pdf', 'subject': '', 'title': '10-K', 'total_pages': 167, 'trapped': ''}, page_content='PART I\nItem 1A\nIssues in the development, deployment, and use of AI may result in reputational or competitive \nharm\n \nor\n \nliability. We are building AI into many of our offerings, including our productivity services, and we are also \nmaking AI available for our customers to use in solutions that they build. This AI may be developed by \nMicrosoft or others, including our strategic partner, OpenAI. We expect these elements of our business to \ngrow. We envision a future in which AI operating in devices, applications, and the cloud helps our  \ncustomers be more productive in their work and personal li

In [ ]:

for i, doc in enumerate(result):
    print(f'\n----Result {i+1}-----\n')
    print(doc.page_content)


----Result 1-----

PART I
Item 1A
Issues in the development, deployment, and use of AI may result in reputational or competitive 
harm
 
or
 
liability. We are building AI into many of our offerings, including our productivity services, and we are also 
making AI available for our customers to use in solutions that they build. This AI may be developed by 
Microsoft or others, including our strategic partner, OpenAI. We expect these elements of our business to 
grow. We envision a future in which AI operating in devices, applications, and the cloud helps our  
customers be more productive in their work and personal lives. As with many innovations, AI presents 
risks and challenges that could affect its adoption, and therefore our business. AI algorithms or training 
methodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate 
information. Content generated by AI systems may be offensive, illegal, inaccurate, or otherwise harmful.

----Result 2-

Contecxtual retreiver

In [ ]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_openai import ChatOpenAI
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_core.prompts import PromptTemplate

model = ChatOpenAI(temperature=0)


retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={'k': 2}
)

compressor = LLMChainExtractor.from_llm(
    llm=model
)

contextual_retriever = ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor=compressor
)

query = "identify sentiments around AI as a growth driver vs a controlled risk"
result = contextual_retriever.invoke(query)  # plain string

print(result)


[Document(metadata={'author': '', 'creationDate': '', 'creationdate': '', 'creator': '', 'file_path': 'download\\10k_pdf\\MSFT-20250730.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': '', 'moddate': '', 'page': 39, 'producer': 'WeasyPrint 66.0', 'source': 'download\\10k_pdf\\MSFT-20250730.pdf', 'subject': '', 'title': '10-K', 'total_pages': 167, 'trapped': ''}, page_content='We envision a future in which AI operating in devices, applications, and the cloud helps our customers be more productive in their work and personal lives. As with many innovations, AI presents risks and challenges that could affect its adoption, and therefore our business.')]


In [ ]:
for i, doc in enumerate(result):
    print(f'\n----Result {i+1}-----\n')
    print(doc.page_content)


----Result 1-----

We envision a future in which AI operating in devices, applications, and the cloud helps our customers be more productive in their work and personal lives. As with many innovations, AI presents risks and challenges that could affect its adoption, and therefore our business.


In [5]:
from config_loader import *  # If you have any global configs
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.chains import RetrievalQA

import os
import re
import argparse
from pathlib import Path
from typing import Dict, Optional
from tqdm import tqdm

# ------------------------------
# CONFIG
# ------------------------------
OPENAI_EMBED_MODEL = "text-embedding-3-large"
OPENAI_CHAT_MODEL = "gpt-4o-mini"

BATCH_SIZE = 50
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

# ------------------------------
# HELPERS
# ------------------------------
def extract_metadata_from_filename(filename: str) -> Dict:
    print("extract_metadata_from_filename")
    basename = os.path.basename(filename)
    print(f"filename: {basename}")
    match = re.match(r"^(.*?)[_-](\d{8})\.pdf$", "DELL-20250325.pdf")
    if match:
        ticker, date_str = match.groups()
        metadata_dict={"ticker": ticker, "date": date_str}
        print(f"Metadata: {metadata_dict}")
        return metadata_dict
    else:
        metadata_dict={"ticker": None, "date": None}
        return metadata_dict


def create_or_load_vectorstore(
    docs=None,
    persist_dir: Path = None,
    collection_name: str = None
):
    print("create_or_load_vectorstore")
    emb = OpenAIEmbeddings(model=OPENAI_EMBED_MODEL)

    if not persist_dir.exists() and docs:
        vectordb = Chroma.from_documents(
            documents=docs,
            embedding=emb,
            persist_directory=str(persist_dir),
            collection_name=collection_name
        )
    else:
        vectordb = Chroma(
            persist_directory=str(persist_dir),
            embedding_function=emb,
            collection_name=collection_name
        )
    return vectordb


def build_chroma_from_dir(pdf_dir: str, persist_dir: str, collection_name: str) -> Chroma:
    print("build_chroma_from_dir")
    if "OPENAI_API_KEY" not in os.environ:
        raise RuntimeError("Set OPENAI_API_KEY before calling this function")

    pdf_dir = Path(pdf_dir)
    files = sorted([p for p in pdf_dir.glob("*.pdf")])
    all_docs = []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, 
        chunk_overlap=CHUNK_OVERLAP
    )    

    for f in tqdm(files, desc="Loading PDFs"):
        print(f"file: {f}")
        loader = PyMuPDFLoader(str(f))
        pages = loader.load_and_split(text_splitter=splitter)
        metadata = extract_metadata_from_filename(f.name)
        for d in pages:
            d.metadata.update({
                "source_file": str(f),
                "ticker": metadata["ticker"],
                "date": metadata["date"]
            })
        all_docs.extend(pages)

    emb = OpenAIEmbeddings(model=OPENAI_EMBED_MODEL)
    vectordb = Chroma.from_documents(
        documents=all_docs,
        embedding=emb,
        persist_directory=persist_dir,
        collection_name=collection_name
    )
    print(f"Collection '{collection_name}' created with {len(all_docs)} documents.")
    return vectordb


def chroma_rag_query(
    query: str,
    vectordb: Chroma,
    persist_dir: str,
    collection_name: str,
    ticker: Optional[str],
    retreiver_search_type: str = "mmr",
    chain_type: str = "map_reduce",
    k: int = 3
):
    print("chroma_rag_query")
    llm = ChatOpenAI(model=OPENAI_CHAT_MODEL, temperature=0)
    print("LLM MODEL INITIALISED")

    template = """
    You are a financial analysis assistant.
    Use ONLY the provided document excerpts to answer the question.
    If the answer is not found in the context, clearly say "Not found in documents."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
    combine_template = """
    Combine the following partial answers into a final answer:
    {summaries}

    Question:
    {question}

    Final Answer:
    """
    refine_template = """
    We have an existing answer:
    {existing_answer}

    We have more context to consider:
    {context}

    Refine the original answer if necessary:
    """
    retriever_prompt = PromptTemplate(
    template="Generate different versions of the following question to improve document retrieval:\n\n{question}",
    input_variables=["question"]
)


    prompt = PromptTemplate(template=template, input_variables=["context", "question"])
    combine_prompt = PromptTemplate(template=combine_template, input_variables=["summaries", "question"])
    refine_prompt = PromptTemplate(template=refine_template, input_variables=["existing_answer", "question"])

    if ticker:
        search_filter = {"ticker": ticker.lower()}  # or .lower() to match your stored metadata

        retriever = vectordb.as_retriever(
        search_type=retreiver_search_type,
        search_kwargs={"k": k, "filter": search_filter}
        )
    else:
        retriever = vectordb.as_retriever(
        search_type=retreiver_search_type,
        search_kwargs={"k": k}
        )
    
    retriver=MultiQueryRetriever.from_llm(
    llm=llm,
    retriever=retriever,
    prompt=retriever_prompt
    )
    
    if chain_type == "stuff":
        chain_kwargs = {"prompt": prompt}
    elif chain_type == "map_reduce":
        chain_kwargs = {"question_prompt": prompt, "combine_prompt": combine_prompt}
    elif chain_type == "refine":
        chain_kwargs = {"question_prompt": prompt, "refine_prompt": refine_prompt}
    else:
        raise ValueError(f"Invalid chain_type '{chain_type}'.")

    print(f'Based on chain type input: {chain_type}: \n {chain_kwargs} ')

    chain = RetrievalQA.from_chain_type(
        retriever=retriver,
        chain_type=chain_type,
        llm=llm,
        chain_type_kwargs=chain_kwargs,
        return_source_documents=True,
        input_key="question"
    )
    result = chain.invoke({"question": query})

    print("Answer:", result["result"])
    print("\nSources:")

    print(result)
    for doc in result["source_documents"]:
        print(f" - {doc.metadata.get('ticker')} ({doc.metadata.get('date')}): {doc.metadata.get('source_file')}")


# ------------------------------
# MAIN
# ------------------------------
def main():
    parser = argparse.ArgumentParser(description="Financial Document RAG Pipeline")
    parser.add_argument("--pdf_dir", type=str, default="download/10k_pdf", help="Path to PDF directory")
    parser.add_argument("--persist_dir", type=str, default="chroma_db_sec", help="Chroma DB persist directory")
    parser.add_argument("--collection_name", type=str, default="10k", help="Chroma collection name")
    parser.add_argument("--query", type=str, default=None, help="User query")
    parser.add_argument("--ticker", type=str, default=None, help="Optional ticker filter")
    parser.add_argument("--chain_type", type=str, default="map_reduce", choices=["stuff", "map_reduce", "refine"], help="Chain type")
    parser.add_argument("--rebuild", action="store_true", help="Rebuild Chroma DB from PDFs")
    parser.add_argument("--k", type=int, default=3, help="Number of docs to retrieve")
    parser.add_argument("--search_type", type=str, default="mmr", choices=["similarity", "mmr"], help="Retriever search type")
    args = parser.parse_args()

    print(20*'#'," STEP1 ", 20*'#')

    print("\n===== Arguments =====")
    for arg, value in vars(args).items():
        print(f"{arg}: {value}")
    print("=====================\n")

    print(20*'#'," STEP2 ", 20*'#')
    if "OPENAI_API_KEY" not in os.environ:
        raise RuntimeError("Please set OPENAI_API_KEY in your environment.")
    else:
        print("OPENAI_API_KEY Is Present in environment")

    print(20*'#'," STEP3 ", 20*'#')
    persist_dir = Path(args.persist_dir)

    if args.rebuild or not persist_dir.exists():
        print("Building Chroma DB...")
        vectordb = build_chroma_from_dir(args.pdf_dir, str(persist_dir), args.collection_name)
    else:
        print("Loading existing Chroma DB...")
        vectordb = create_or_load_vectorstore(
            docs=None,
            persist_dir=persist_dir,
            collection_name=args.collection_name
        )

    print(20*'#'," STEP4 ", 20*'#')

    if args.query:
        query=args.query
    else:
        query="identify sentiments around AI as a growth driver vs a controlled risk"

    if args.ticker:
        ticker=args.ticker
    else:
        ticker="MMM"
    print("executing chroma_rag_query...")
    chroma_rag_query(
        query=query,
        vectordb=vectordb,
        persist_dir=str(persist_dir),
        collection_name=args.collection_name,
        ticker=ticker,
        retreiver_search_type=args.search_type,
        chain_type=args.chain_type,
        k=args.k
    )
